# AI Marketing Strategy Manager
### A Multi-Agent System built with the OpenAI Agents SDK

Domain: **Marketing**



## 1. Problem Analysis

**Business context**

Marketing teams juggle a lot at once: watching competitors, spotting trends, planning campaigns, writing content, reading analytics dashboards, and deciding what to fix next. Doing all of this by hand is slow, and the work is scattered across different tools and spreadsheets, so insights often arrive too late to act on.

**Stakeholders**
- Marketing Manager / CMO — owns strategy and final campaign approval
- Content & Social Media team — needs a ready content calendar
- Performance Marketing / Growth team — needs budget allocation and channel guidance
- Analytics team — needs consolidated performance reporting
- Leadership / Finance — approves budget above a set threshold

**Problem statement**

Marketing teams need a single assistant that can research the market, track competitors, plan and budget campaigns, draft content, measure performance, and recommend the next optimisation — while keeping a human in the loop for any decision that spends real money.

**Objectives**
1. Automate competitor and market-trend research
2. Turn research into a structured, budgeted campaign plan
3. Generate a content calendar aligned to the campaign
4. Summarise campaign performance from analytics data
5. Recommend concrete optimisations, with human approval required above a budget threshold
6. Keep memory of the conversation so the team doesn't repeat context every turn

## 2. Multi-Agent Design

**Agent architecture** (manager + specialists, with one specialist also acting as a sub-manager over three others)

```
                         ┌───────────────────────────┐
                         │   Marketing Manager        │   <- Triage / entry point
                         │   (handoff router)         │
                         └─────────────┬──────────────┘
        ┌───────────────┬──────────────┼───────────────┬───────────────┐
        ▼                ▼             ▼                ▼               ▼
 Market Research   Competitor     Campaign Planner  Content       Optimisation
     Agent         Analysis Agent      Agent       Strategist       Advisor
                                                       Agent      (uses the 3 agents
                                                                   below AS TOOLS)
                                                                        │
                                            ┌───────────────────────────┼───────────────┐
                                            ▼                           ▼               ▼
                                     Market Research(tool)     Competitor(tool)   Analytics Agent
```

**Roles of each agent**

| Agent | Role |
|---|---|
| Marketing Manager (Triage) | Understands the request and hands off to the right specialist |
| Market Research Agent | Looks up industry trends and market size/growth signals |
| Competitor Analysis Agent | Looks up a named competitor's positioning, pricing, channels |
| Campaign Planner Agent | Produces a structured, budgeted campaign plan |
| Content Strategist Agent | Produces a structured content calendar for the campaign |
| Analytics Agent | Reads campaign performance data and produces a structured report |
| Optimisation Advisor | Manager pattern — calls Market Research, Competitor Analysis and Analytics as tools, then recommends next steps, flagging anything over budget for human approval |

**Agent interaction and handoff flow**
1. User talks to the **Marketing Manager**.
2. An **input guardrail** checks the request is actually marketing-related before anything runs.
3. The Manager **hands off** to whichever specialist matches the request (research, competitor, planning, content, analytics, optimisation).
4. The **Optimisation Advisor** doesn't hand off — instead it calls three other agents *as tools* (`as_tool()`), combines their outputs, and returns one recommendation.
5. If the Optimisation Advisor's recommendation changes the budget beyond a threshold, `human_approval_required` is set on the structured output and a **human approval step** gates execution.

**Tool integration overview**

| Tool | Used by | Purpose |
|---|---|---|
| `get_market_trends` | Market Research Agent | Look up trend/growth data for an industry |
| `get_competitor_profile` | Competitor Analysis Agent | Look up a competitor's profile |
| `calculate_campaign_budget` | Campaign Planner Agent | Split a total budget across channels |
| `get_campaign_performance` | Analytics Agent | Look up performance metrics for a campaign |
| `get_current_date` | Campaign Planner Agent | Timestamp the plan / compute campaign dates |
| `log_event` | multiple agents | Structured logging / error handling |
| `market_research_agent.as_tool()`, `competitor_analysis_agent.as_tool()`, `analytics_agent.as_tool()` | Optimisation Advisor | Manager pattern — agents used as callable tools |

## 3. Implementation

### 3.1 Setup

In [1]:
!pip -q install -U openai-agents pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.5/941.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.3 MB/s eta 0:00:00


In [2]:
import importlib.metadata as metadata
import sys

print("Python:", sys.version)
print("OpenAI Agents:", metadata.version("openai-agents"))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OpenAI Agents: 0.19.2


In [3]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

In [4]:
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Paste your OpenAI key from: https://platform.openai.com/account/api-keys\n"
    )

print("API key found")

API key found


In [5]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("marketing_agents")

In [6]:
from dataclasses import dataclass, field

@dataclass
class BrandContext:
    brand_name: str
    industry: str
    monthly_budget: float
    target_audience: str
    approval_threshold: float = 5000.0   # spend above this needs human sign-off

brand_context = BrandContext(
    brand_name="Nimbus Fitness",
    industry="fitness apps",
    monthly_budget=12000,
    target_audience="urban professionals aged 25-40",
)

print(brand_context)

BrandContext(brand_name='Nimbus Fitness', industry='fitness apps', monthly_budget=12000, target_audience='urban professionals aged 25-40', approval_threshold=5000.0)


In [7]:
MARKET_TRENDS_DB = {
    "fitness apps": {
        "growth_rate": "14% YoY",
        "top_trend": "AI-personalised workout plans",
        "audience_shift": "growing interest from 35-50 age group",
        "hot_channels": ["TikTok", "Instagram Reels", "YouTube Shorts"],
    },
    "sustainable fashion": {
        "growth_rate": "9% YoY",
        "top_trend": "resale and rental platforms",
        "audience_shift": "Gen Z prioritising traceability over price",
        "hot_channels": ["Instagram", "Pinterest", "TikTok"],
    },
    "saas productivity tools": {
        "growth_rate": "11% YoY",
        "top_trend": "AI copilots embedded in existing workflows",
        "audience_shift": "buyers now individual contributors, not just IT",
        "hot_channels": ["LinkedIn", "YouTube", "Newsletters"],
    },
}

COMPETITOR_DB = {
    "flexfit": {
        "positioning": "budget fitness app with large free tier",
        "pricing": "$0-$9.99/mo",
        "main_channels": ["TikTok", "Influencer partnerships"],
        "recent_move": "launched AI form-correction feature",
    },
    "pulseup": {
        "positioning": "premium, coach-led fitness app",
        "pricing": "$29.99/mo",
        "main_channels": ["Instagram", "YouTube", "Podcast sponsorships"],
        "recent_move": "signed two celebrity trainers",
    },
}

CAMPAIGN_PERFORMANCE_DB = {
    "spring-launch": {
        "impressions": 480000,
        "clicks": 9600,
        "ctr_percent": 2.0,
        "conversions": 384,
        "conversion_rate_percent": 4.0,
        "spend": 6400,
        "cac": 16.67,
    },
    "summer-push": {
        "impressions": 210000,
        "clicks": 3150,
        "ctr_percent": 1.5,
        "conversions": 63,
        "conversion_rate_percent": 2.0,
        "spend": 5800,
        "cac": 92.06,
    },
}

### Tools



In [8]:
from agents import function_tool, RunContextWrapper

@function_tool
def get_market_trends(industry: str) -> str:
    """Return current market trend data for an industry.

    Args:
        industry: Industry name, e.g. "fitness apps" or "sustainable fashion".
    """
    data = MARKET_TRENDS_DB.get(industry.strip().lower())
    if not data:
        return f"No trend data found for industry: {industry}"
    return (
        f"Growth rate: {data['growth_rate']}\n"
        f"Top trend: {data['top_trend']}\n"
        f"Audience shift: {data['audience_shift']}\n"
        f"Hot channels: {', '.join(data['hot_channels'])}"
    )

In [9]:
@function_tool
def get_competitor_profile(competitor_name: str) -> str:
    """Return a competitor's positioning, pricing and channel profile.

    Args:
        competitor_name: Name of the competitor, e.g. "FlexFit".
    """
    profile = COMPETITOR_DB.get(competitor_name.strip().lower())
    if not profile:
        return f"No profile found for competitor: {competitor_name}"
    return (
        f"Positioning: {profile['positioning']}\n"
        f"Pricing: {profile['pricing']}\n"
        f"Main channels: {', '.join(profile['main_channels'])}\n"
        f"Recent move: {profile['recent_move']}"
    )

In [10]:
@function_tool
def calculate_campaign_budget(total_budget: float, channels: list[str]) -> str:
    """Split a total campaign budget evenly across a list of channels.

    Args:
        total_budget: Total amount available for the campaign.
        channels: Channel names to split the budget across, e.g. ["TikTok", "Instagram"].
    """
    if total_budget <= 0 or not channels:
        return "total_budget must be positive and at least one channel must be given."

    per_channel = total_budget / len(channels)
    lines = [f"{ch}: ${per_channel:,.2f}" for ch in channels]
    return "Budget allocation:\n" + "\n".join(lines)

In [11]:
@function_tool
def get_campaign_performance(campaign_name: str) -> str:
    """Return performance metrics for a named campaign.

    Args:
        campaign_name: Campaign identifier, e.g. "spring-launch".
    """
    metrics = CAMPAIGN_PERFORMANCE_DB.get(campaign_name.strip().lower())
    if not metrics:
        return f"No performance data found for campaign: {campaign_name}"
    return (
        f"Impressions: {metrics['impressions']:,} | Clicks: {metrics['clicks']:,} "
        f"| CTR: {metrics['ctr_percent']}%\n"
        f"Conversions: {metrics['conversions']} | Conversion rate: {metrics['conversion_rate_percent']}%\n"
        f"Spend: ${metrics['spend']:,} | CAC: ${metrics['cac']:.2f}"
    )

In [12]:
from datetime import datetime
from zoneinfo import ZoneInfo

@function_tool
def get_current_date(timezone_name: str = "Asia/Kolkata") -> str:
    """Return today's date for an IANA timezone, useful for scheduling campaigns.

    Args:
        timezone_name: IANA timezone such as Asia/Kolkata or America/New_York.
    """
    try:
        now = datetime.now(ZoneInfo(timezone_name))
        return now.strftime("%Y-%m-%d")
    except Exception:
        return f"Unknown timezone: {timezone_name}"

In [13]:
@function_tool
def log_event(agent_name: str, event: str) -> str:
    """Log an event raised by an agent, e.g. a completed lookup or a flagged risk.

    Args:
        agent_name: Name of the agent raising the event.
        event: Short description of what happened.
    """
    try:
        logger.info(f"[{agent_name}] {event}")
        return "Logged."
    except Exception as exc:
        logger.error(f"Logging failed: {exc}")
        return "Failed to log event."

In [14]:
from pydantic import BaseModel, Field
from typing import Literal

class CampaignPlan(BaseModel):
    campaign_name: str
    objective: str
    target_audience: str
    channels: list[str] = Field(min_length=1, max_length=6)
    total_budget: float
    budget_allocation: str = Field(description="Per-channel budget breakdown")
    start_date: str
    key_message: str

class ContentItem(BaseModel):
    day: str
    channel: str
    content_type: Literal["video", "image_post", "carousel", "story", "blog"]
    topic: str

class ContentCalendar(BaseModel):
    campaign_name: str
    items: list[ContentItem] = Field(min_length=3, max_length=14)

class PerformanceReport(BaseModel):
    campaign_name: str
    summary: str
    ctr_percent: float
    conversion_rate_percent: float
    cac: float
    verdict: Literal["under-performing", "on-target", "over-performing"]

class OptimizationRecommendation(BaseModel):
    headline_recommendation: str
    reasoning: str
    proposed_budget_change: float = Field(description="Positive to increase spend, negative to cut it")
    human_approval_required: bool = Field(
        description="True if proposed_budget_change exceeds the brand's approval threshold"
    )

### 3 Specialist agents

Five research/execution specialists, each with a role, tools, and (where useful) a structured `output_type`. Instructions follow Role → Process → Constraints → Output Format, same as the intro notebook's "better instructions" pattern.

In [15]:
from agents import Agent, Runner, SQLiteSession

MODEL_NAME = "gpt-4o-mini"

market_research_agent = Agent(
    name="Market Research Agent",
    instructions="""
    You are a market research specialist.
    Process: use get_market_trends for the requested industry, then summarise the growth
    rate, top trend, audience shift and best channels in plain language.
    Constraints: never invent numbers that were not returned by the tool.
    Output format: a short paragraph followed by a 3-4 bullet summary.
    """,
    handoff_description="Researches industry trends, growth rates and audience shifts.",
    model=MODEL_NAME,
    tools=[get_market_trends, log_event],
)

In [16]:
competitor_analysis_agent = Agent(
    name="Competitor Analysis Agent",
    instructions="""
    You are a competitor intelligence specialist.
    Process: use get_competitor_profile for the named competitor(s), then compare their
    positioning, pricing and channels against the brand.
    Constraints: if a competitor is not found, say so clearly instead of guessing.
    Output format: one short paragraph per competitor, ending with a one-line takeaway.
    """,
    handoff_description="Looks up and compares named competitors' positioning, pricing and channels.",
    model=MODEL_NAME,
    tools=[get_competitor_profile, log_event],
)

In [17]:
def campaign_planner_instructions(ctx: RunContextWrapper[BrandContext], agent: Agent[BrandContext]) -> str:
    brand = ctx.context
    return f"""
    You are a campaign planning specialist for {brand.brand_name}, in the {brand.industry} industry.
    Default target audience unless told otherwise: {brand.target_audience}.
    Default total budget unless told otherwise: ${brand.monthly_budget:,.0f}.

    Process: pick 2-4 channels that fit the audience, use calculate_campaign_budget to split
    the total budget across them, and use get_current_date for the start date.
    Constraints: budget_allocation must reflect the tool output, not a guess.
    Output format: return the CampaignPlan structured object.
    """

campaign_planner_agent = Agent[BrandContext](
    name="Campaign Planner Agent",
    instructions=campaign_planner_instructions,
    handoff_description="Builds a structured, budgeted campaign plan.",
    model=MODEL_NAME,
    tools=[calculate_campaign_budget, get_current_date, log_event],
    output_type=CampaignPlan,
)

In [18]:
content_strategist_agent = Agent(
    name="Content Strategist Agent",
    instructions="""
    You are a content strategist.
    Process: given a campaign name, channels and key message, design a short content
    calendar (one week is enough unless asked for more).
    Constraints: vary the content_type across items, don't repeat the same topic twice.
    Output format: return the ContentCalendar structured object.
    """,
    handoff_description="Turns a campaign plan into a structured content calendar.",
    model=MODEL_NAME,
    output_type=ContentCalendar,
)

In [19]:
analytics_agent = Agent(
    name="Analytics Agent",
    instructions="""
    You are a marketing analytics specialist.
    Process: use get_campaign_performance for the named campaign, then judge the verdict:
    "under-performing" if conversion_rate_percent < 3, "over-performing" if > 5, else "on-target".
    Constraints: base every number on the tool output only.
    Output format: return the PerformanceReport structured object.
    """,
    handoff_description="Reads campaign performance data and reports on it.",
    model=MODEL_NAME,
    tools=[get_campaign_performance, log_event],
    output_type=PerformanceReport,
)

In [20]:
def optimisation_instructions(ctx: RunContextWrapper[BrandContext], agent: Agent[BrandContext]) -> str:
    brand = ctx.context
    return f"""
    You are the optimisation advisor for {brand.brand_name}.
    Any proposed_budget_change with an absolute value greater than ${brand.approval_threshold:,.0f}
    MUST have human_approval_required set to true. Otherwise set it to false.

    Process: call research_market_trends and research_competitor as needed, then
    analyze_campaign_performance for the named campaign, and combine all three into one
    recommendation.
    Output format: return the OptimizationRecommendation structured object.
    """

optimisation_advisor_agent = Agent[BrandContext](
    name="Optimisation Advisor",
    instructions=optimisation_instructions,
    handoff_description="Combines research, competitor and analytics insight into one recommendation.",
    model=MODEL_NAME,
    tools=[
        market_research_agent.as_tool(
            tool_name="research_market_trends",
            tool_description="Ask the market research agent for industry trend data.",
        ),
        competitor_analysis_agent.as_tool(
            tool_name="research_competitor",
            tool_description="Ask the competitor analysis agent about a named competitor.",
        ),
        analytics_agent.as_tool(
            tool_name="analyze_campaign_performance",
            tool_description="Ask the analytics agent for a campaign's performance report.",
        ),
        log_event,
    ],
    output_type=OptimizationRecommendation,
)

###  Guardrails: keeping the Manager on-topic

An input guardrail runs a small classifier agent before the main request proceeds, and trips a tripwire if the request has nothing to do with marketing.

In [21]:
from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, TResponseInputItem
from agents.decorators import input_guardrail

class RelevanceCheck(BaseModel):
    is_marketing_related: bool
    reason: str

relevance_checker = Agent(
    name="Marketing Relevance Checker",
    instructions="""
    Classify whether the request is related to marketing strategy, research, campaigns,
    content, analytics or optimisation. Unrelated small talk or off-topic requests should
    be marked as not marketing related.
    """,
    output_type=RelevanceCheck,
    model=MODEL_NAME,
)

@input_guardrail
async def marketing_relevance_guardrail(
    ctx: RunContextWrapper[BrandContext], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    check_result = await Runner.run(relevance_checker, input, context=ctx.context)
    classification = check_result.final_output
    return GuardrailFunctionOutput(
        output_info=classification,
        tripwire_triggered=not classification.is_marketing_related,
    )

###  Marketing Manager — the triage agent (handoffs)

This is the single entry point the user talks to. It routes the request to the right specialist via **handoff**, and carries the `marketing_relevance_guardrail` on its input.

In [22]:
marketing_manager = Agent[BrandContext](
    name="Marketing Manager",
    instructions="""
    You are the triage point for a marketing strategy team.
    Identify what the user needs and hand off to exactly one specialist:
    - Market Research Agent for industry trends
    - Competitor Analysis Agent for a named competitor
    - Campaign Planner Agent to plan or budget a campaign
    - Content Strategist Agent to build a content calendar
    - Analytics Agent to read performance data
    - Optimisation Advisor for a "what should we do next" style question
    If the request spans more than one need, hand off to the most central one first.
    """,
    model=MODEL_NAME,
    handoffs=[
        market_research_agent,
        competitor_analysis_agent,
        campaign_planner_agent,
        content_strategist_agent,
        analytics_agent,
        optimisation_advisor_agent,
    ],
    input_guardrails=[marketing_relevance_guardrail],
)

In [23]:
def request_human_approval(recommendation: "OptimizationRecommendation") -> bool:
    """Pause and ask a human to approve a budget-changing recommendation."""
    print("\n--- HUMAN APPROVAL REQUIRED ---")
    print("Recommendation:", recommendation.headline_recommendation)
    print("Reasoning:", recommendation.reasoning)
    print(f"Proposed budget change: ${recommendation.proposed_budget_change:,.2f}")
    answer = input("Approve this change? (yes/no): ").strip().lower()
    approved = answer == "yes"
    logger.info(f"Human approval decision: {approved}")
    return approved

###  Memory & session persistence



In [24]:
session = SQLiteSession("nimbus_fitness_marketing_thread")

In [25]:
async def run_manager(prompt: str):
    """Run one turn through the Marketing Manager, with logging and error handling."""
    try:
        result = await Runner.run(
            marketing_manager,
            prompt,
            session=session,
            context=brand_context,
        )
        print(f"[Handled by: {result.last_agent.name}]\n")
        print(result.final_output)
        return result
    except InputGuardrailTripwireTriggered:
        print("Request blocked by the marketing-relevance guardrail.")
    except Exception as exc:
        logger.error(f"Run failed: {exc}")
        print("Something went wrong — see the log above.")

In [26]:
# Turn 1: market research
await run_manager("What are the current market trends for fitness apps?");

[Handled by: Market Research Agent]

The fitness app market is currently experiencing a robust growth rate of 14% year-over-year, highlighting its increasing popularity. A significant trend is the adoption of AI-personalised workout plans, which cater to individual user needs and preferences. Notably, there's been a shift in the audience demographic, with more interest coming from the 35 to 50 age group. The best channels for engagement and marketing these fitness apps include TikTok, Instagram Reels, and YouTube Shorts.

- **Growth Rate**: 14% year-over-year.
- **Top Trend**: AI-personalised workout plans.
- **Audience Shift**: Increasing interest from the 35-50 age group.
- **Best Channels**: TikTok, Instagram Reels, YouTube Shorts.


In [27]:
# Turn 2: competitor analysis
await run_manager("How does FlexFit compare to us on pricing and channels?");

[Handled by: Competitor Analysis Agent]

FlexFit positions itself as a budget fitness app, offering a large free tier to attract users. Its pricing ranges from $0 to $9.99 per month, making it accessible for a wide audience. FlexFit primarily uses TikTok and influencer partnerships as its main channels for marketing and engagement, aligning with current trends in social media. Recently, they've introduced an AI form-correction feature, enhancing user experience and engagement.

- **Positioning**: Budget fitness app with a large free tier.
- **Pricing**: $0-$9.99 per month.
- **Main Channels**: TikTok, influencer partnerships.

**Takeaway**: FlexFit's budget-friendly pricing and strong social media presence make it a formidable competitor.


In [28]:
# Turn 3: campaign planning (structured output)
plan_result = await run_manager(
    "Plan a campaign called Autumn Reset using TikTok, Instagram and YouTube."
)

[Handled by: Campaign Planner Agent]

campaign_name='Autumn Reset' objective='Engage users with personalized fitness plans for the autumn season.' target_audience='urban professionals aged 25-40.' channels=['TikTok', 'Instagram', 'YouTube'] total_budget=12000.0 budget_allocation='TikTok: $4,000, Instagram: $4,000, YouTube: $4,000' start_date='2026-08-02' key_message='Rejuvenate your fitness journey this autumn with tailored plans!'


In [29]:
# Inspect the structured CampaignPlan returned by the Campaign Planner
campaign_plan = plan_result.final_output
print(type(campaign_plan))
print(campaign_plan.model_dump_json(indent=2))

<class '__main__.CampaignPlan'>
{
  "campaign_name": "Autumn Reset",
  "objective": "Engage users with personalized fitness plans for the autumn season.",
  "target_audience": "urban professionals aged 25-40.",
  "channels": [
    "TikTok",
    "Instagram",
    "YouTube"
  ],
  "total_budget": 12000.0,
  "budget_allocation": "TikTok: $4,000, Instagram: $4,000, YouTube: $4,000",
  "start_date": "2026-08-02",
  "key_message": "Rejuvenate your fitness journey this autumn with tailored plans!"
}


In [30]:
# Turn 4: content calendar
await run_manager(
    "Build a content calendar for the Autumn Reset campaign on TikTok and Instagram, "
    "key message: small consistent workouts beat big ones you skip."
);

[Handled by: Content Strategist Agent]

campaign_name='Autumn Reset' items=[ContentItem(day='Monday', channel='TikTok', content_type='video', topic='Importance of consistent workouts'), ContentItem(day='Tuesday', channel='Instagram', content_type='image_post', topic='Visualization of short workout routines'), ContentItem(day='Wednesday', channel='TikTok', content_type='story', topic='User testimonials on daily mini workouts'), ContentItem(day='Thursday', channel='Instagram', content_type='carousel', topic='5 quick workouts for busy professionals'), ContentItem(day='Friday', channel='TikTok', content_type='video', topic='Myth-busting: Small workouts vs. large workouts'), ContentItem(day='Saturday', channel='Instagram', content_type='blog', topic='Benefits of building a workout habit'), ContentItem(day='Sunday', channel='TikTok', content_type='story', topic='Weekly recap and motivation for next week!')]


In [31]:
# Turn 5: analytics
await run_manager("How did the spring-launch campaign perform?");

[Handled by: Analytics Agent]

campaign_name='spring-launch' summary='The spring-launch campaign engaged a substantial audience with a total of 480,000 impressions and a click-through rate of 2.0%. It achieved 384 conversions, leading to a conversion rate of 4.0%.' ctr_percent=2.0 conversion_rate_percent=4.0 cac=16.67 verdict='on-target'


In [32]:
# Turn 6: optimisation advice, then gate on human approval if needed
opt_result = await run_manager(
    "Based on our market position, FlexFit, and the spring-launch results, "
    "what should we do next with our budget?"
)

recommendation = opt_result.final_output
if recommendation.human_approval_required:
    if request_human_approval(recommendation):
        print("\nApproved — proceeding with the budget change.")
    else:
        print("\nNot approved — no budget change applied.")
else:
    print("\nWithin threshold — no human approval needed.")

[Handled by: Optimisation Advisor]

headline_recommendation='Increase Budget for targeted AI features and Expanded Channel Use.' reasoning="The market is growing at 14% YoY, and FlexFit's positioning highlights the importance of personalization and affordability. Given our spring-launch performance has shown decent engagement but modest conversions, allocating more budget towards AI features and expanding reach on TikTok and Instagram could significantly enhance our competitive edge and attract the key demographic (35-50 age group). This alignment with market trends and competitor positioning could maximize returns on our investment." proposed_budget_change=8000.0 human_approval_required=True

--- HUMAN APPROVAL REQUIRED ---
Recommendation: Increase Budget for targeted AI features and Expanded Channel Use.
Reasoning: The market is growing at 14% YoY, and FlexFit's positioning highlights the importance of personalization and affordability. Given our spring-launch performance has shown d

In [33]:
# Turn 7: memory check — the manager should recall earlier context from this session
await run_manager("Remind me which campaign we just planned and which channels it uses.");

[Handled by: Marketing Manager]

The campaign we just planned is called **Autumn Reset**. It utilizes the following channels:

- TikTok
- Instagram
- YouTube


In [34]:
# Off-topic request — the relevance guardrail should block this before any agent runs
await run_manager("Can you write me a poem about the ocean?");

Request blocked by the marketing-relevance guardrail.
